# 6. Modelo Macro (M3, Entrega 3)

Este notebook junta dados macroeconômicos públicos à base e mede se eles melhoram o modelo. O regulamento pede a fonte de cada dado, o método para levar o dado (do país ou do estado) até a linha de cada cliente e como o modelo rodaria em produção, sabendo que o dado do governo sai com atraso.

Regras desta etapa, definidas antes de rodar:

- **M3 = M2 + colunas macro.** O M2 é o M1 mais a família de CEP (`reports/variaveis_m2.json`). O LightGBM usa os parâmetros do M1 (`reports/params_m1.json`), fixos: aqui o modelo é só o instrumento de medida.
- **Sem informação do futuro:** cada concessão é tratada como feita no dia 1 do mês da safra, e cada série só vale depois da sua defasagem de divulgação (`docs/macro_catalogo.md`).
- **Validação temporal:** treino nas safras 202301–202306 e avaliação em 202307–202308. No teste dos notebooks 03 e 04, treino e teste têm os mesmos meses; uma série com um valor por mês pode servir para o modelo reconhecer o mês e decorar a inadimplência dele. Com meses novos na avaliação, isso não ajuda.
- **Duas versões:** todas as séries (16 colunas) e só as que variam entre clientes do mesmo mês (desocupação da UF, a variação dela em 12 meses e a diferença para o Brasil). As colunas de idade do dado ficam fora do modelo; servem para monitorar atrasos em produção.
- **Regra de decisão (definida antes de rodar):** fica a versão com maior AUC no protocolo temporal, se o ganho sobre o M2 superar o desvio entre sementes. Se nenhuma superar, o M3 fica com a versão só com UF, que não tem como decorar o mês e depende de uma fonte só.

## 6.1 Imports e leitura

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from equifax.config import RANDOM_SEED, REPORTS, TABLES
from equifax.conjuntos import carregar_conjunto
from equifax.dados import carregar_desenvolvimento, janelas
from equifax.experimentos import comparar_com_referencia, ler_registro, rodar_experimento, treinar
from equifax.explicar import contribuicoes, importancia_shap
from equifax.features import Cep, gerar_features
from equifax.macro import CATALOGO, juntar_macro, montar_tabela_macro, pasta_mais_recente
from equifax.tratamento import SEM_INFORMACAO
from equifax.validacao import separar_janelas
from equifax.viz import salvar_figura

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
PARAMS_M1 = json.load(open(REPORTS / "params_m1.json", encoding="utf-8"))
M2 = json.load(open(REPORTS / "variaveis_m2.json", encoding="utf-8"))
CONJUNTO = PARAMS_M1["conjunto"]
PARAMS_LGBM = PARAMS_M1["lgbm"]
VARIAVEIS_M1 = carregar_conjunto(CONJUNTO)
VARIAVEIS_M2 = M2["variaveis"]
assert M2["base"] == CONJUNTO, "o M2 precisa ter sido gerado sobre o mesmo M1"
assert M2["familias"] == ["f4_cep"], "este notebook gera só a família de CEP; rode o 04 de novo se o M2 mudou"
print(CONJUNTO, "| M2:", len(VARIAVEIS_M2), "variáveis, famílias", M2["familias"])

base = carregar_desenvolvimento()
dev, _ = separar_janelas(base)  # 202301-202308; a OOT interna não entra
safra = dev["safra"].astype(int)

## 6.2 As séries e a regra de disponibilidade

Oito séries do Banco Central (SGS) e do IBGE (SIDRA), baixadas com a data do download e um manifesto (`data/external/macro_bruto/`). A defasagem é o número de meses entre o mês de referência e o primeiro dia em que o valor pode ser usado; ela cobre a data normal de divulgação com folga.

In [ ]:
print("Download usado:", pasta_mais_recente().name)
pd.DataFrame([{"serie": s.nome, "fonte": f"{s.fonte.upper()} {s.codigo}", "unidade": s.unidade,
               "frequencia": s.frequencia, "geografia": s.geografia,
               "defasagem_meses": s.defasagem_meses if s.frequencia != "diaria" else "dia seguinte"}
              for s in CATALOGO.values()]).set_index("serie")

In [ ]:
tabela = montar_tabela_macro()
macro = juntar_macro(dev, tabela)
print(macro.shape[1], "colunas macro;", int(macro.isna().sum().sum()), "valores ausentes")

SERIES = [c for c in macro.columns if not c.endswith("_idade_meses")]
SERIES_UF = ["macro_desocupacao_uf", "macro_desocupacao_uf_var_12m", "macro_desocupacao_uf_menos_br"]

# Exemplo da regra: o mês de referência que cada série usa na safra 202307
idade = macro[[c for c in macro.columns if c.endswith("_idade_meses")]].groupby(safra).max()
exemplo = idade.loc[202307].rename("idade_meses").to_frame()
exemplo.index = exemplo.index.str.removeprefix("macro_").str.removesuffix("_idade_meses")
exemplo["referencia_usada"] = [(pd.Period("2023-07", "M") - int(m)).strftime("%m/%Y") for m in exemplo["idade_meses"]]
exemplo

## 6.3 O que esperar dos dados macro

Análise de se a inadimplência da carteira acompanha a economia mês a mês e se estados com mais desemprego têm mais inadimplência

In [ ]:
# Valores que o modelo enxerga em cada safra (séries nacionais: um valor por mês)
nacionais = ["macro_selic_meta", "macro_ipca_12m", "macro_inadimplencia_pf", "macro_desocupacao_br",
             "macro_comprometimento_renda", "macro_endividamento_familias"]
por_safra = macro[nacionais].groupby(safra).first()
por_safra.columns = por_safra.columns.str.removeprefix("macro_")
por_safra.insert(0, "inadimplencia_carteira_%", 100 * (1 - dev["adimplencia"]).groupby(safra).mean())
por_safra.round(2)

In [ ]:
# Quantos valores diferentes cada coluna tem nas safras de treino (202301-202306)
treino_t = safra <= 202306
valores_diferentes = macro.loc[treino_t, SERIES].nunique().sort_values()
valores_diferentes.index = valores_diferentes.index.str.removeprefix("macro_")
valores_diferentes.rename("valores_no_treino").to_frame().T

In [ ]:
# Por estado, nas safras de treino: inadimplência da carteira x desocupação da UF
estado = dev["estado"].astype(str)
por_uf = (pd.DataFrame({"estado": estado, "mau": 1 - dev["adimplencia"],
                        "desocupacao": macro["macro_desocupacao_uf"]})[treino_t]
          .query("estado != @SEM_INFORMACAO")
          .groupby("estado").agg(clientes=("mau", "size"), inadimplencia=("mau", "mean"),
                                 desocupacao=("desocupacao", "mean")))
correlacao = por_uf["inadimplencia"].corr(por_uf["desocupacao"], method="spearman")

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(por_uf["desocupacao"], 100 * por_uf["inadimplencia"], s=por_uf["clientes"] / 150,
           alpha=.6, color="steelblue")
for uf, linha in por_uf.iterrows():
    ax.annotate(uf, (linha["desocupacao"], 100 * linha["inadimplencia"]), fontsize=7,
                xytext=(3, 2), textcoords="offset points")
ax.set_xlabel("Desocupação da UF disponível na concessão (%, média 202301-202306)")
ax.set_ylabel("Inadimplência da carteira (%)")
ax.set_title(f"Estados com mais desemprego têm mais inadimplência? (Spearman {correlacao:.2f})")
ax.grid(alpha=.3)
salvar_figura("m3_desocupacao_uf_x_inadimplencia")
plt.show()
print(f"{len(por_uf)} estados; tamanho do ponto = número de clientes")

## 6.4 Validação temporal: M2 contra M3

Os modelos são ajustados nas safras 202301–202306 e avaliados em 202307–202308; o M1 entra só como referência. A família de CEP aprende a taxa por prefixo só no treino. As colunas macro entram como estão, com os valores disponíveis no dia 1 de cada safra

In [ ]:
partes_t = {"treino": dev[treino_t], "temporal": dev[~treino_t]}
partes_t, colunas_cep = gerar_features(partes_t, [Cep(suavizacao=200)])
for k in partes_t:
    partes_t[k] = pd.concat([partes_t[k], macro.loc[partes_t[k].index, SERIES]], axis=1)
for k, p in partes_t.items():
    print(f"{k:8} n={len(p):7} safras {p['safra'].min()}-{p['safra'].max()} "
          f"inadimplência={1 - p['adimplencia'].mean():.4f}")

EXPERIMENTOS_T = {"m3_t_m1": VARIAVEIS_M1,
                  "m3_t_m2": VARIAVEIS_M2,
                  "m3_t_mais_macro_uf": VARIAVEIS_M2 + SERIES_UF,
                  "m3_t_mais_macro_todas": VARIAVEIS_M2 + SERIES}
for nome, variaveis in EXPERIMENTOS_T.items():
    rodar_experimento(nome, variaveis, partes_t, modelo="lgbm", params=PARAMS_LGBM,
                      avaliar_em="temporal", nome_conjunto=f"{CONJUNTO}+{nome.removeprefix('m3_t_')}")
    print(nome, "pronto")

temporal = comparar_com_referencia(ler_registro(), "m3_t_m2",
                                   ["m3_t_m1", "m3_t_mais_macro_uf", "m3_t_mais_macro_todas"])
temporal.to_csv(TABLES / "m3_temporal.csv")
temporal

## 6.5 A mesma comparação no teste por cliente

Mesmas variáveis, mas na divisão dos notebooks 03 e 04 (treino e teste nas mesmas safras, 202301–202308). Se o macro parecer ajudar aqui e não na validação temporal, o ganho vem de reconhecer o mês ao invés de medir risco

In [ ]:
partes = janelas(base)
partes, _ = gerar_features(partes, [Cep(suavizacao=200)])
for k in partes:
    partes[k] = pd.concat([partes[k], macro.loc[partes[k].index, SERIES]], axis=1)

EXPERIMENTOS = {"m3_m2": VARIAVEIS_M2,
                "m3_mais_macro_uf": VARIAVEIS_M2 + SERIES_UF,
                "m3_mais_macro_todas": VARIAVEIS_M2 + SERIES}
for nome, variaveis in EXPERIMENTOS.items():
    rodar_experimento(nome, variaveis, partes, modelo="lgbm", params=PARAMS_LGBM,
                      nome_conjunto=f"{CONJUNTO}+{nome.removeprefix('m3_')}")
    print(nome, "pronto")

mesmo_periodo = comparar_com_referencia(ler_registro(), "m3_m2", ["m3_mais_macro_uf", "m3_mais_macro_todas"])
mesmo_periodo.to_csv(TABLES / "m3_teste_por_cliente.csv")
mesmo_periodo

## 6.6 Uso das colunas macro pelo modelo

Contribuições SHAP do M3 com todas as séries, treinado em 202301–202306, numa amostra de 10 mil linhas de 202307–202308. A tabela mostra a posição de cada coluna macro entre todas as variáveis e quanto do efeito total vem delas

In [ ]:
m3_todas = treinar("lgbm", PARAMS_LGBM, partes_t["treino"], VARIAVEIS_M2 + SERIES, semente=RANDOM_SEED)
contrib, valor_base, x = contribuicoes(m3_todas, partes_t["temporal"], amostra=10000, semente=RANDOM_SEED)
imp = importancia_shap(contrib)
posicao = imp.rank(ascending=False).astype(int)
uso_macro = pd.DataFrame({"media_abs_shap": imp, "posicao": posicao}).loc[[c for c in imp.index if c in SERIES]]
print(f"Parte do efeito total que vem das {len(SERIES)} colunas macro: {imp[SERIES].sum() / imp.sum():.1%}")
uso_macro.round(4)

## 6.7 Decisão do M3

Aplica a regra definida no começo do notebook e grava a lista de variáveis do M3 (só nomes de colunas)

In [ ]:
candidatas = temporal.drop(index=["m3_t_m2", "m3_t_m1"])
ganhou = candidatas[candidatas["supera_ruido"] == True]
if len(ganhou):
    escolhida = ganhou["auc"].idxmax()
else:
    escolhida = "m3_t_mais_macro_uf"
SERIES_M3 = SERIES_UF if escolhida == "m3_t_mais_macro_uf" else SERIES
VARIAVEIS_M3 = VARIAVEIS_M2 + SERIES_M3
print("Versão escolhida:", escolhida, "|", len(VARIAVEIS_M3), "variáveis")

with open(REPORTS / "variaveis_m3.json", "w", encoding="utf-8") as f:
    json.dump({"base": CONJUNTO, "m2": "variaveis_m2.json", "versao": escolhida,
               "series_macro": [c.removeprefix("macro_") for c in SERIES_M3],
               "variaveis": VARIAVEIS_M3}, f, ensure_ascii=False, indent=2)

## 6.8 Em produção: que dado o modelo usaria na OOT

O modelo em produção recebe, para cada concessão, o último valor de cada série já divulgado no dia 1 do mês. A tabela mostra o que entra nas safras da OOT cega (novembro e dezembro de 2023), calculado só a partir das datas, sem abrir a OOT. Se uma divulgação atrasar, o valor anterior continua valendo e a idade do dado aumenta; a escoragem registra essa idade por safra.

In [ ]:
datas_oot = pd.DataFrame({"safra": [202311, 202312], "estado": ["São Paulo", "São Paulo"]})
na_oot = juntar_macro(datas_oot, tabela)
idade_oot = na_oot[[c for c in na_oot.columns if c.endswith("_idade_meses")]].T
idade_oot.columns = ["202311", "202312"]
idade_oot.index = idade_oot.index.str.removeprefix("macro_").str.removesuffix("_idade_meses")
for s in ["202311", "202312"]:
    idade_oot[f"referencia_{s}"] = [(pd.Period(f"{s[:4]}-{s[4:]}", "M") - int(m)).strftime("%m/%Y")
                                    for m in idade_oot[s]]
idade_oot

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.